#파이썬 버전

In [ ]:
import sys
import os
import platform

# Python 버전 확인
print("Python Version:", sys.version)

# 현재 작업 디렉토리 확인
print("Current Working Directory:", os.getcwd())

# 운영 체제 및 아키텍처 확인
print("OS Platform:", platform.system())
print("OS Architecture:", platform.machine())

# 설치된 패키지 목록 확인
!pip list

#카메라 테스트

In [ ]:
#카메라 촬영
import cv2
from pop import Camera
from pop import Util

# 카메라 초기화 (300x300 해상도)
cam = Camera(300, 300)

# imshow 활성화
Util.enable_imshow()

# 전역 변수로 프레임 저장
global_frame = None

while True:
    # 카메라에서 프레임 읽기
    img = cam.value

    # 전역 변수에 프레임 저장
    global_frame = img

    # 현재 프레임을 "Camera Feed" 창에 표시
    Util.imshow("Camera Feed", img)

    # 키 입력 대기 (1ms)
    key = cv2.waitKey(1) & 0xFF

    # 'q' 키를 누르면 종료
    if key == ord('q'):
        break

# 자원 해제 및 창 닫기
cv2.destroyAllWindows()

#카메라 구현 성공 전역변수 송출 성공, multiprocessing.Value 이용

In [ ]:
#카메라 실시간 구동, 시험 준비 완료
import cv2
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import time
from pop import Camera
from IPython.display import display
import ipywidgets as widgets
from threading import Thread
from multiprocessing import Value
import ctypes

# 디바이스 설정 (GPU 사용 가능 여부 확인)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 전역 변수: 상태 값 (0: free, 1: blocked)
# 공유 메모리 객체를 사용하여 상태 저장
status_shared = Value(ctypes.c_int, 0)  # 초기값은 free 상태

# 모델 정의
class MyTransferLearningModel(torch.nn.Module):
    def __init__(self, pretrained_model, feature_extractor):
        super().__init__()
        if feature_extractor:
            for param in pretrained_model.parameters():
                param.requires_grad = False
        pretrained_model.classifier = torch.nn.Sequential(
            torch.nn.Linear(pretrained_model.classifier[1].in_features, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.5),
            torch.nn.Linear(64, 16),
            torch.nn.ReLU(),
            torch.nn.Linear(16, 2)  # 2가지 분류 (blocked, free)
        )
        self.model = pretrained_model

    def forward(self, data):
        logits = self.model(data)
        return logits

# 사전 학습된 MobileNetV2 모델 로드
pretrained_model = models.mobilenet_v2(pretrained=True)

# Feature Extractor 여부 설정
feature_extractor = True

# 모델 인스턴스 생성
model = MyTransferLearningModel(pretrained_model, feature_extractor).to(DEVICE)

# 저장된 가중치 파일 경로 (Jetson 환경에 맞게 수정)
MODEL_PATH = "/home/soda/Project/python/notebook/dataset2/MobileNetV2_best_model3.pth"

# 모델 가중치 로드
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()  # 모델을 평가 모드로 설정

# 입력 이미지 전처리
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # MobileNetV2는 224x224 입력을 기대합니다
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet 정규화
])

# 클래스 레이블 정의
class_labels = ["blocked", "free"]

# 상태 값을 숫자로 변환하는 함수
def get_status_value(state):
    return 1 if state == "blocked" else 0

# 상태 업데이트 함수
def update_status(frame):
    global last_state, current_state, state_start_time, status_shared

    # 모델 입력을 위한 전처리
    pil_image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))  # OpenCV 이미지를 PIL 이미지로 변환
    input_tensor = transform(pil_image).unsqueeze(0).to(DEVICE)  # 배치 차원 추가 및 GPU로 이동

    # 모델 예측
    with torch.no_grad():  # 그래디언트 계산 비활성화
        outputs = model(input_tensor)
        _, predicted = torch.max(outputs, 1)  # 가장 높은 확률의 클래스 선택

    # 예측 결과 가져오기
    result = class_labels[predicted.item()]

    # 상태 변화 체크
    if result != current_state:
        current_state = result
        state_start_time = time.time()  # 상태 시작 시간 갱신

    # 상태가 유지된 시간 계산
    elapsed_time = time.time() - state_start_time

    # 상태가 0.5초 이상 유지되었을 때만 최종 결과로 간주
    if elapsed_time >= STATE_DURATION_THRESHOLD:
        last_state = current_state

    # 상태 값을 숫자로 변환하고 공유 메모리에 업데이트 (0: free, 1: blocked)
    status_value = get_status_value(last_state)
    with status_shared.get_lock():
        status_shared.value = status_value

    return status_value, last_state

# 카메라 초기화 (224x224 해상도)
cam = Camera(224, 224)

# 상태와 시간 초기화
last_state = "free"  # 이전 상태 초기값 설정
current_state = "free"  # 현재 상태 초기값 설정
state_start_time = time.time()  # 상태 시작 시간
STATE_DURATION_THRESHOLD = 0.5  # 상태가 유지되어야 하는 최소 시간 (초)

# Jupyter Notebook에서 이미지를 표시하기 위한 위젯
image_widget = widgets.Image(format='jpeg', width=224, height=224)
display(image_widget)

# 트리거 상태 초기화
trigger_active = False  # 트리거 활성화 여부
last_trigger_time = time.time()  # 마지막 트리거 출력 시간

# 버튼 위젯 생성
trigger_button = widgets.ToggleButton(
    value=False,
    description="Trigger",
    button_style="success",  # 'success', 'info', 'warning', 'danger' 중 선택
    tooltip="Toggle trigger on/off"
)

# 상태 출력 텍스트 위젯 생성
status_output = widgets.Text(
    value='Status: 0 (free)',
    description='현재 상태:',
    disabled=True
)
display(status_output)

# 버튼 클릭 시 트리거 상태 변경
def on_button_click(change):
    global trigger_active
    trigger_active = change.new
    if trigger_active:
        print("트리거 활성화됨 (1초마다 상태 출력)")
    else:
        print("트리거 비활성화됨")

trigger_button.observe(on_button_click, names="value")
display(trigger_button)

print("실시간 처리 시작...")

# 현재 상태값을 가져오는 함수 정의 (다른 셀에서 사용 가능)
def get_current_status():
    with status_shared.get_lock():
        return status_shared.value

def camera_loop():
    global trigger_active, last_trigger_time
    while True:
        # 현재 프레임 읽기
        frame = cam.value
        if frame is None:
            print("프레임을 읽을 수 없습니다.")
            break

        # 상태 업데이트 및 값 반환
        status_value, last_state = update_status(frame)

        # 상태를 화면 상단에 표시
        cv2.putText(frame, f"Status: {last_state} ({status_value})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        # 상태 텍스트 위젯 업데이트
        status_output.value = f'Status: {status_value} ({last_state})'

        # Jupyter Notebook에서 프레임 표시
        image_widget.value = cv2.imencode('.jpg', frame)[1].tobytes()

        # 트리거 상태에 따른 자동 출력
        if trigger_active and time.time() - last_trigger_time >= 1.0:  # 1초마다 상태 출력
            current_status = get_current_status()
            print(f"현재 상태: {current_status} ({last_state})")
            last_trigger_time = time.time()

# 별도 스레드에서 카메라 루프 실행
camera_thread = Thread(target=camera_loop)
camera_thread.daemon = True
camera_thread.start()

In [ ]:
import cv2
import torch # DEVICE 설정에만 사용, 실제 YOLO 추론은 OpenCV DNN 사용
import numpy as np
import time
from pop import Camera
from IPython.display import display
import ipywidgets as widgets
from threading import Thread
from multiprocessing import Value # 감지된 객체 수 등을 공유하기 위해 유지
import ctypes
import os # 파일 경로 확인용

# --- 설정값 ---
# !!! 사용자의 환경에 맞게 경로를 수정해주세요 !!!
WEIGHTS_PATH = "/home/soda/Project/python/notebook/yolo_files/custom-train-yolo4_best.weights" # 예시 경로
CFG_PATH = "/home/soda/Project/python/notebook/yolo_files/custom-test-yolo4.cfg" # 예시 경로
NAMES_PATH = "/home/soda/Project/python/notebook/yolo_files/classes.names" # 예시 경로

# YOLO 입력 크기 (CFG 파일의 [net] 섹션 width, height 와 일치해야 함)
# 일반적인 YOLOv4 값 중 하나를 사용하거나 CFG 파일에서 직접 확인하세요.
INPUT_WIDTH = 416
INPUT_HEIGHT = 416

CONF_THRESHOLD = 0.5  # 객체 탐지 최소 신뢰도
NMS_THRESHOLD = 0.4   # Non-Maximum Suppression 임계값

# 디바이스 설정 (OpenCV DNN은 CUDA를 지원)
# PyTorch의 DEVICE 설정은 여기서는 OpenCV DNN 백엔드 선택에 간접적으로만 참고합니다.
USE_CUDA = torch.cuda.is_available()
print(f"CUDA Available: {USE_CUDA}")

# 공유 메모리: 감지된 객체의 수를 저장 (예시)
# 좀 더 복잡한 정보를 공유하려면 구조를 변경해야 합니다.
detected_object_count_shared = Value(ctypes.c_int, 0)

# 클래스 이름 로드
try:
    with open(NAMES_PATH, "r") as f:
        class_names = [line.strip() for line in f.readlines()]
except FileNotFoundError:
    print(f"오류: {NAMES_PATH} 파일을 찾을 수 없습니다. 경로를 확인하세요.")
    class_names = [] # 빈 리스트로 초기화

colors = np.random.uniform(0, 255, size=(len(class_names), 3)) # 각 클래스별 바운딩 박스 색상

# 모델 로드 (OpenCV DNN 사용)
net = None
try:
    if not os.path.exists(WEIGHTS_PATH):
        raise FileNotFoundError(f"Weights 파일 없음: {WEIGHTS_PATH}")
    if not os.path.exists(CFG_PATH):
        raise FileNotFoundError(f"CFG 파일 없음: {CFG_PATH}")

    net = cv2.dnn.readNetFromDarknet(CFG_PATH, WEIGHTS_PATH)
    print("YOLOv4 모델 로드 성공")

    if USE_CUDA:
        net.setPreferableBackend(cv2.dnn.DNN_BACKEND_CUDA)
        net.setPreferableTarget(cv2.dnn.DNN_TARGET_CUDA)
        print("OpenCV DNN 백엔드를 CUDA로 설정했습니다.")
    else:
        net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
        print("OpenCV DNN 백엔드를 CPU로 설정했습니다.")

    # 출력 레이어 이름 가져오기
    layer_names = net.getLayerNames()
    # 이전 OpenCV 버전에서는 getUnconnectedOutLayers()가 정수 리스트를 반환
    # 최신 버전에서는 레이어 이름을 직접 가져오거나, 인덱스를 통해 이름을 찾아야 할 수 있음
    try:
        # OpenCV 4.x.x 이상
        output_layer_indexes = net.getUnconnectedOutLayers()
        if isinstance(output_layer_indexes, np.ndarray) and output_layer_indexes.ndim > 1: # [[idx1], [idx2]] 형태
             output_layer_names = [layer_names[i[0] - 1] for i in output_layer_indexes]
        elif isinstance(output_layer_indexes, np.ndarray) and output_layer_indexes.ndim == 1: # [idx1, idx2] 형태
             output_layer_names = [layer_names[i - 1] for i in output_layer_indexes]
        else: # 단일 인덱스 경우 (거의 없음)
             output_layer_names = [layer_names[output_layer_indexes - 1]]

    except AttributeError:
        # OpenCV 3.x.x
        output_layer_names = [layer_names[i[0] - 1] for i in net.getUnconnectedOutLayers()]
    print(f"출력 레이어: {output_layer_names}")


except Exception as e:
    print(f"모델 로드 중 오류 발생: {e}")
    # net이 None이면 이후 코드에서 문제를 일으키므로, 적절히 처리 필요
    # 여기서는 실행 중단 대신 print만 하고 넘어감. 실제로는 exit() 등을 고려.

# 카메라 초기화 (224x224 해상도) -> YOLO 입력 크기에 맞춰 조정하거나, 내부에서 리사이즈
cam = Camera(width=max(INPUT_WIDTH, 224), height=max(INPUT_HEIGHT, 224)) # 카메라 해상도는 YOLO 입력보다 크거나 같게

# Jupyter Notebook에서 이미지를 표시하기 위한 위젯
image_widget = widgets.Image(format='jpeg', width=INPUT_WIDTH, height=INPUT_HEIGHT) # 표시 크기
display(image_widget)

# 트리거 상태 및 출력 제어 (이전 코드와 유사하게 유지)
trigger_active = False
last_trigger_time = time.time()

trigger_button = widgets.ToggleButton(
    value=False,
    description="Detections Log",
    button_style="info",
    tooltip="Toggle detection logging"
)

detection_output_widget = widgets.Textarea(
    value='',
    placeholder='감지된 객체 정보가 여기에 표시됩니다.',
    description='감지 로그:',
    disabled=False,
    layout={'width': '90%', 'height': '100px'}
)

def on_button_click(change):
    global trigger_active
    trigger_active = change.new
    if trigger_active:
        print("객체 감지 로그 활성화됨 (감지 시마다 출력)")
        detection_output_widget.value = "로그 활성화됨\n"
    else:
        print("객체 감지 로그 비활성화됨")
        detection_output_widget.value = "로그 비활성화됨\n"

trigger_button.observe(on_button_click, names="value")
display(trigger_button)
display(detection_output_widget)


# 객체 탐지 및 바운딩 박스 그리기 함수
def process_frame_yolo(frame):
    if net is None or not class_names:
        if frame is not None:
             # 모델 로드 실패 시 원본 프레임에 메시지 표시
            cv2.putText(frame, "YOLO Model NOT LOADED", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        return frame, 0, "모델 로드 실패"

    H, W = frame.shape[:2]

    # 이미지 전처리 및 블롭 생성
    # swapRB=True: OpenCV는 BGR, YOLO는 RGB를 기대하므로 채널 순서 변경
    blob = cv2.dnn.blobFromImage(frame, 1/255.0, (INPUT_WIDTH, INPUT_HEIGHT), swapRB=True, crop=False)
    net.setInput(blob)

    # 추론 실행
    detections = net.forward(output_layer_names) # detections는 output_layers의 결과 리스트

    boxes = []
    confidences = []
    class_ids = []

    # 모든 탐지 결과 반복
    for output in detections:
        for detection in output:
            scores = detection[5:] # 클래스별 신뢰도
            class_id = np.argmax(scores)
            confidence = scores[class_id]

            if confidence > CONF_THRESHOLD:
                # 바운딩 박스 좌표 계산 (원본 이미지 비율로)
                # YOLO는 중심 x, 중심 y, 너비, 높이를 0~1 사이 값으로 반환
                box = detection[0:4] * np.array([W, H, W, H])
                (centerX, centerY, width, height) = box.astype("int")

                # 좌상단 좌표 계산
                x = int(centerX - (width / 2))
                y = int(centerY - (height / 2))

                boxes.append([x, y, int(width), int(height)])
                confidences.append(float(confidence))
                class_ids.append(class_id)

    # Non-Maximum Suppression 적용
    indices = cv2.dnn.NMSBoxes(boxes, confidences, CONF_THRESHOLD, NMS_THRESHOLD)

    detected_info_log = ""
    num_detected_objects = 0

    if len(indices) > 0:
        num_detected_objects = len(indices)
        for i in indices.flatten(): # indices는 [[idx1], [idx2]] 형태일 수 있음
            (x, y) = (boxes[i][0], boxes[i][1])
            (w, h) = (boxes[i][2], boxes[i][3])

            color = colors[class_ids[i]]
            cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
            text = f"{class_names[class_ids[i]]}: {confidences[i]:.2f}"
            cv2.putText(frame, text, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            detected_info_log += f"  - {class_names[class_ids[i]]} (Conf: {confidences[i]:.2f}) at [{x},{y},{w},{h}]\n"

    # 공유 메모리에 감지된 객체 수 업데이트
    with detected_object_count_shared.get_lock():
        detected_object_count_shared.value = num_detected_objects

    return frame, num_detected_objects, detected_info_log


print("실시간 객체 탐지 시작...")

def camera_loop():
    global trigger_active, last_trigger_time, detection_output_widget
    fps_start_time = time.time()
    fps_counter = 0

    while True:
        frame = cam.value
        if frame is None:
            print("프레임을 읽을 수 없습니다.")
            time.sleep(0.1) # 잠시 대기 후 재시도
            continue

        # 프레임 복사 ( 원본 유지하며 처리하기 위함 )
        processed_frame = frame.copy()

        # YOLO 객체 탐지
        processed_frame, num_detections, detection_log = process_frame_yolo(processed_frame)

        # FPS 계산
        fps_counter += 1
        if (time.time() - fps_start_time) > 1: # 1초마다 FPS 업데이트
            fps = fps_counter / (time.time() - fps_start_time)
            cv2.putText(processed_frame, f"FPS: {fps:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            fps_start_time = time.time()
            fps_counter = 0
        else: # 이전 FPS 값 표시 (깜빡임 방지)
             # (옵션) 이전 FPS 값을 저장해두고 표시하거나, 매번 계산하지 않아도 됨
             pass


        # 상태를 화면 상단에 표시 (감지된 객체 수)
        status_text = f"Detected: {num_detections}"
        cv2.putText(processed_frame, status_text, (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)


        # Jupyter Notebook에서 프레임 표시
        # 표시 위젯 크기에 맞게 리사이즈 (선택적, 화질 저하 가능성 있음)
        display_frame = cv2.resize(processed_frame, (INPUT_WIDTH, INPUT_HEIGHT))
        image_widget.value = cv2.imencode('.jpg', display_frame)[1].tobytes()

        # 트리거 상태에 따른 자동 출력
        if trigger_active and num_detections > 0: # 감지된 객체가 있을 때만 로그 업데이트
            current_time_str = time.strftime("%Y-%m-%d %H:%M:%S")
            log_entry = f"[{current_time_str}] {num_detections} objects detected:\n{detection_log}"
            # 위젯에 너무 많은 텍스트가 쌓이는 것을 방지하기 위해 최근 로그만 유지 (예시)
            current_log = detection_output_widget.value.splitlines()
            max_log_lines = 10
            if len(current_log) > max_log_lines:
                detection_output_widget.value = "\n".join(current_log[-(max_log_lines-1):]) + "\n" + log_entry
            else:
                detection_output_widget.value += log_entry + "\n"


        # 루프 지연 (너무 빠르게 돌면 CPU 사용량 증가)
        # time.sleep(0.01) # 필요에 따라 조절

# 별도 스레드에서 카메라 루프 실행
camera_thread = Thread(target=camera_loop)
camera_thread.daemon = True # 메인 프로그램 종료 시 스레드도 함께 종료
camera_thread.start()

# 현재 감지된 객체 수를 가져오는 함수 (다른 셀에서 사용 가능)
def get_current_detection_count():
    with detected_object_count_shared.get_lock():
        return detected_object_count_shared.value